# HTML 解析与兼容

学习目标：能对照源码与 DOM 解释常见解析差异，区分 HTML、XML 和文档模式，并判断旧式标记的迁移方向。

前置知识：HTML 元素嵌套、列表与表格，浏览器开发者工具，JavaScript 变量、函数调用和条件判断。

适用范围：WHATWG HTML Living Standard、XML 1.0 第五版；使用支持 DOMParser 的现代浏览器。示例保留明确标注的错误输入，不作为新页面写法。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/17-parsing-and-compatibility/。

1. [dom.html](scripts/17-parsing-and-compatibility/dom.html)、[dom-cases.js](scripts/17-parsing-and-compatibility/dom-cases.js)：完整文档树、标签省略与段落错误恢复。
2. [whitespace.html](scripts/17-parsing-and-compatibility/whitespace.html)：空白文本节点与排版。
3. [xml.html](scripts/17-parsing-and-compatibility/xml.html)：同一输入的 HTML/XML 解析与 XML 预期失败。
4. [mime-probe.xhtml](scripts/17-parsing-and-compatibility/mime-probe.xhtml)、[preview_server.py](scripts/17-parsing-and-compatibility/preview_server.py)：同一文件的两种 Content-Type 响应。
5. [standards.html](scripts/17-parsing-and-compatibility/standards.html)、[quirks.html](scripts/17-parsing-and-compatibility/quirks.html)：有、无 DOCTYPE 的模式对照。
6. [migration.html](scripts/17-parsing-and-compatibility/migration.html)：旧式标记及迁移后的结构、样式。

## 打开配套页面

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/html
```

Step 2：启动本章预览服务。

```bash
python scripts/17-parsing-and-compatibility/preview_server.py
```

Step 3：打开[解析示例入口](http://127.0.0.1:8017/dom.html)。

服务绑定本机 127.0.0.1 的 8017 端口

本章专用服务以 scripts/17-parsing-and-compatibility 为服务根目录，所以浏览器路径是 /dom.html，不再重复 scripts/17-parsing-and-compatibility。Notebook 的配套文件链接仍从 Notebook 所在目录计算。

两个 MIME 实验地址由服务明确设置 Content-Type，不能用普通静态服务或直接打开磁盘文件代替。所有素材均在本章目录；浏览器脚本只处理固定输入和显示读数。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 从源码观察文档树

“查看源代码”显示收到的 HTML 文本，Elements 显示当前 DOM。文档对象模型（Document Object Model，DOM）用节点和父子关系表达文档；元素树只关注元素，完整 DOM 还含文本、注释等节点。

HTML 解析先识别标签与文本等标记，再按树构建规则建立文档。合法省略、错误恢复都可能让 DOM 与标签表面写法不同；脚本还可能在解析后修改 DOM。

dom.html 的脚本把下面固定字符串作为独立文档解析：

```html
<!doctype html><title>阅读记录</title><p>开始学习。</p>
```

配套文件：[scripts/17-parsing-and-compatibility/dom.html](scripts/17-parsing-and-compatibility/dom.html) · [浏览器预览](http://127.0.0.1:8017/dom.html)

这个输入满足 &lt;html&gt;、&lt;head&gt;、&lt;body&gt; 的标签省略条件，省略标签后对应元素仍存在。浏览器 API DOMParser 在此选择 text/html，并返回单独的内存文档：

```javascript
const parsed = new DOMParser().parseFromString(source, "text/html");
// 本例 parsed 的 html 下有 head 和 body；head 中是 title，body 中是 p。
```

配套文件：[scripts/17-parsing-and-compatibility/dom.html](scripts/17-parsing-and-compatibility/dom.html) · [浏览器预览](http://127.0.0.1:8017/dom.html)

source 是上述源码字符串，parsed 是解析后的 Document。页面通过 outerHTML 把树序列化为文字，再用 textContent 显示；序列化不保留原文件的逐字写法，也没有把解析出的树插入当前页面。

## 2 合法省略：标签不在，元素仍在

### 2.1 列表项结束标签

&lt;li&gt; 后紧接另一个 &lt;li&gt;，或已到父元素内容末尾时，可省略结束标签。下面是 dom-cases.js 第一组 source 的原文；完整解析输入还会补上 DOCTYPE 和文档标题。

```html
<ul><li>阅读<li>整理</ul>
```

配套文件：[scripts/17-parsing-and-compatibility/dom-cases.js](scripts/17-parsing-and-compatibility/dom-cases.js) · [浏览器预览](http://127.0.0.1:8017/dom.html)

### 2.2 表格的隐含表体

下面表格的 &lt;tbody&gt; 起止标签满足省略条件，解析树仍包含 &lt;tbody&gt;。

```html
<table><tr><td>周六</td></tr></table>
```

配套文件：[scripts/17-parsing-and-compatibility/dom-cases.js](scripts/17-parsing-and-compatibility/dom-cases.js) · [浏览器预览](http://127.0.0.1:8017/dom.html)

这里首项是 &lt;tr&gt;，前面没有紧邻且省略了结束标签的表体或表头，表体后也没有其他内容。不能把这个特例推广为任何位置的标签都可省略，也不能据此推断脚本手工创建的表格树。

## 3 错误恢复：浏览器能显示不等于合规

检查错误恢复时，先预测节点之间的关系，再看文字是否显示。&lt;p&gt; 不能包含 &lt;div&gt;；解析器遇到这里的 &lt;div&gt; 起始标签会先结束段落，后方失配的 &lt;/p&gt; 又触发恢复，产生空段落。

![反例源码被解析为 body 下四个兄弟节点：含开头的 p、含独立内容的 div、结尾文本和空 p。](image/illustration/17-01-parser-recovered-tree.svg)

图 1：紧邻代码片段的恢复结果。第三项是文本节点，不是段落；连线表示真实父子关系，未画文档其他节点。

在配套页同时看 childNodes 与 children：前者能看见“结尾”文本，后者只列元素。修改反例前先决定“结尾”应该归到哪个段落。

```html
<p>开头<div>独立内容</div>结尾</p>
```

配套文件：[scripts/17-parsing-and-compatibility/dom-cases.js](scripts/17-parsing-and-compatibility/dom-cases.js) · [浏览器预览](http://127.0.0.1:8017/dom.html)

这个输入是明确的反例。在 &lt;div&gt; 前省略 &lt;p&gt; 结束标签本身可以合法，问题不能概括成“段落不能省略结束标签”。修复时先决定“结尾”应属于哪个段落，再写清同级结构。

- parse error：解析算法明确指出的语法错误，标准规定其处理方式。
- content model：内容模型，规定元素允许的内容；检查范围不限于解析错误。

配套页比较三组固定输入与预期树，只用于观察这些例子；得到 DOM 或比较值为 true，不代表完成 HTML 合规校验。

## 4 空白节点与空白排版

普通元素内的空格、换行通常成为文本节点。CSS 折叠空白影响显示，不会因此删除 DOM 中的文本。

以下两段来自 whitespace.html，文字完全相同，配套 CSS 只改变 white-space：

```html
<p id="normal" lang="en">Read   HTML
then   compare.</p>
```

```html
<p id="preserved" lang="en">Read   HTML
then   compare.</p>
```

配套文件：[scripts/17-parsing-and-compatibility/whitespace.html](scripts/17-parsing-and-compatibility/whitespace.html) · [浏览器预览](http://127.0.0.1:8017/whitespace.html)

- white-space: normal：连续空白按常规规则折叠。
- white-space: pre-wrap：保留空白，同时允许自动换行。
- textContent：读取文本内容，不按视觉排版折叠空格。

下面两个 &lt;span&gt; 之间的换行是独立文本节点。childNodes 包含所有直接子节点，children 只包含直接子元素。

```html
<div id="nodes"><span>Read</span>
<span>HTML</span></div>
```

配套文件：[scripts/17-parsing-and-compatibility/whitespace.html](scripts/17-parsing-and-compatibility/whitespace.html) · [浏览器预览](http://127.0.0.1:8017/whitespace.html)

配套脚本让换行以转义形式显示，并比较两个计数。本例使用英文单词，避免把某种语言的换行显示当成普遍规则。

补充：源码空白并非全部原样保留。HTML 解析会把 CR、CRLF 换行规范化为 LF；紧接 &lt;pre&gt; 起始标签的首个换行有特殊处理，一些文档结构位置的空白也会被忽略。

## 5 HTML 与 XML 的语法差异

HTML 的 XML 写法常称为 XHTML：使用 XML 语法，并让 HTML 元素处于 HTML 命名空间。两种解析规则不同，HTML 不能简单理解为“宽松版 XML”。

- 名称：HTML 语法中的 HTML 元素和属性名称按 ASCII 大小写不敏感处理；XML 名称区分大小写，起止标签须匹配。
- 属性值：HTML 满足限制时可不加引号；XML 属性值必须加引号。
- 布尔属性：HTML 可只写 disabled；XML 不能省略属性值，HTML 的 XML 写法可用 disabled="disabled"。
- 结束标签：HTML 只在指定条件下省略；XML 非空元素必须有匹配的结束标签。
- 空元素：HTML 的 &lt;br&gt; 等空元素本来就没有结束标签；&lt;div&gt; 不是 HTML 空元素，尾部斜线不会使其自行结束。XML 中无内容的元素可用 /&gt; 结束。

上述 HTML 规则针对 HTML 命名空间；SVG、MathML 有外来内容规则，不能一概套用。

xml.html 把这段缺少结束标签的固定字符串分别交给两个解析器：

```html
<ul><li>阅读<li>整理</ul>
```

配套文件：[scripts/17-parsing-and-compatibility/xml.html](scripts/17-parsing-and-compatibility/xml.html) · [浏览器预览](http://127.0.0.1:8017/xml.html)

以下是实际解析调用，missingEnd 保存上述字符串：

```javascript
const htmlList = parser.parseFromString(missingEnd, "text/html");
const badXml = parser.parseFromString(missingEnd, "application/xml");
const xmlError = badXml.querySelector("parsererror");
// htmlList 中有两个 li；xmlError 非 null，具体错误文字随浏览器变化。
```

配套文件：[scripts/17-parsing-and-compatibility/xml.html](scripts/17-parsing-and-compatibility/xml.html) · [浏览器预览](http://127.0.0.1:8017/xml.html)

XML 语法失败通常以返回文档中的 &lt;parsererror&gt; 表示，不应只等待 JavaScript 异常。配套页显式检查预期失败，并提供补齐两个 &lt;/li&gt; 的修复输入；错误文字可能随浏览器变化，不比较完整报错文本。

XML 格式良好（well-formed）只说明语法满足相应要求，不等于符合 HTML 内容模型。DOMParser 也不是自动清除不安全内容的工具。

### 5.1 同一个斜线，不同的树

xml.html 的第二组输入使用 HTML 命名空间，仍由 parseFromString 的类型参数选择解析器。

```html
<section xmlns="http://www.w3.org/1999/xhtml"><div id="box"/><p id="after">后续段落</p></section>
```

配套文件：[scripts/17-parsing-and-compatibility/xml.html](scripts/17-parsing-and-compatibility/xml.html) · [浏览器预览](http://127.0.0.1:8017/xml.html)

- text/html：&lt;div /&gt; 不自行结束，后面的段落成为它的子元素。
- application/xml：空 &lt;div /&gt; 已结束，段落与它同属 &lt;section&gt;。

页面先检查 XML 输入格式良好，再显示两个父元素名称。下面一节把同样的区别用于真实 HTTP 导航，区分“字符串解析参数”与“服务器响应类型”。

## 6 HTTP 响应类型与文件扩展名

MIME 类型表示资源的媒体类型。本章服务显式发送 Content-Type，以相同字节比较两种处理方式，不讨论缺失或错误类型时的内容嗅探。

- [按 HTML 打开](http://127.0.0.1:8017/as-html.xhtml)：响应类型为 text/html，虽然地址以 .xhtml 结尾。
- [按 XML 打开](http://127.0.0.1:8017/as-xml.html)：响应类型为 application/xhtml+xml，虽然地址以 .html 结尾。

两个地址都读取 mime-probe.xhtml，不是两份磁盘文件。下面是该文件 &lt;body&gt; 内的对照片段；根 &lt;html&gt; 已声明 HTML 命名空间。

```html
<main><div id="box"/><p id="after">后续段落</p></main>
```

配套文件：[scripts/17-parsing-and-compatibility/mime-probe.xhtml](scripts/17-parsing-and-compatibility/mime-probe.xhtml) · [浏览器预览](http://127.0.0.1:8017/as-html.xhtml)

在 Network 查看页面请求的 Content-Type，再比较页面的 contentType、parentName 读数，并在 Elements 定位 id 为 after 的段落。

添加 xmlns、XML 风格斜线、DOCTYPE 或改扩展名，都不能替代响应类型来切换解析器。服务只提供字节和响应头，实际父子关系由浏览器解析形成。

## 7 标准模式与怪异模式

新写的 HTML 页面在开头使用简短 DOCTYPE，以启用标准模式。它与选择 HTML/XML 解析器、检查内容是否合规是不同问题。

下面是 standards.html 的开头：

```html
<!doctype html>
<html lang="zh-CN">
```

配套文件：[scripts/17-parsing-and-compatibility/standards.html](scripts/17-parsing-and-compatibility/standards.html) · [浏览器预览](http://127.0.0.1:8017/standards.html)

quirks.html 仅删除第一行，其余内容相同：

```html
<html lang="zh-CN">
  <head>
```

配套文件：[scripts/17-parsing-and-compatibility/quirks.html](scripts/17-parsing-and-compatibility/quirks.html) · [浏览器预览](http://127.0.0.1:8017/quirks.html)

- no-quirks mode：标准模式，document.compatMode 为 CSS1Compat。
- limited-quirks mode：有限怪异模式，某些历史 DOCTYPE 会触发，compatMode 同样为 CSS1Compat。
- quirks mode：怪异模式，为旧页面保留兼容行为，compatMode 为 BackCompat。本例缺少 DOCTYPE 的普通 text/html 完整页面会进入此模式。

compatMode 只能区分怪异与非怪异，不能单独分清标准和有限怪异。通过 application/xhtml+xml 提供的页面使用标准模式，不依赖这个 HTML 模式开关。

两页外观可能相同，需比较实际读数。修复旧页面时，在副本添加 DOCTYPE 后仍要检查布局和交互，防止遗漏依赖旧行为的样式。

## 8 阅读旧式标记并迁移

浏览器保留兼容行为，不代表旧写法仍适合新页面。WHATWG 区分“过时但仍合规”与“不合规”：前者可能产生校验警告，例如普通脚本上冗余的 type="text/javascript"；后者不应用于新写内容。

- &lt;center&gt;：旧式居中元素。按内容选择容器或标题，外观交给 CSS。
- &lt;font&gt;：旧式字体表现元素。字体与颜色交给 CSS。
- &lt;acronym&gt;：旧式缩略词元素，改用 &lt;abbr&gt;。
- &lt;p&gt; 的 align 属性：旧式对齐属性，改用 CSS 的 text-align。

migration.html 将旧源码转义后作为文字显示，迁移后的内容为：

```html
<h3 class="notice-title">阅读提醒</h3>
<p class="notice-text">比较 <abbr title="Document Object Model">DOM</abbr> 与源码。</p>
<!-- 检查：提醒是 h3；缩略词是 abbr；元素树中没有 center、font、acronym。 -->
```

配套文件：[scripts/17-parsing-and-compatibility/migration.html](scripts/17-parsing-and-compatibility/migration.html) · [浏览器预览](http://127.0.0.1:8017/migration.html)

这里提醒属于页面中“迁移后的内容”一节，因此用 &lt;h3&gt;。下面 CSS 只负责水平对齐和颜色：

```css
.notice-title { text-align: center; color: #175cd3; }
.notice-text { text-align: center; }
```

配套文件：[scripts/17-parsing-and-compatibility/migration.html](scripts/17-parsing-and-compatibility/migration.html) · [浏览器预览](http://127.0.0.1:8017/migration.html)

不要按“看起来粗体或斜体”机械替换元素：&lt;b&gt;、&lt;i&gt; 仍有规定语义。&lt;b&gt; 表示引起注意但不增加重要性的文字，&lt;i&gt; 表示不同语气或性质的文字；&lt;strong&gt; 表示重要性，&lt;em&gt; 表示重音强调。

迁移时先核对响应类型和 DOCTYPE，再按内容含义替换标记，最后检查结构、链接与显示。普通新页面沿用 HTML 语法，不因本章 XML 对照而改写为 XHTML。

## 本章小结

- 标签省略和错误恢复都会影响源码与 DOM 的对应关系，但两者的合规含义不同。
- 空白节点与视觉空白分开观察，元素数量不等于全部节点数量。
- DOMParser 由参数选择解析器，HTTP 导航则看响应类型；扩展名和 DOCTYPE 都不能替代 MIME 判断。
- compatMode 不足以区分标准与有限怪异，也不表示 HTML 已校验通过。
- 迁移旧标记时保留内容含义，把表现交给 CSS。

自查：一条结论来自源代码、HTTP 响应、解析树，还是当前页面？

## 练习

在 scripts/17-parsing-and-compatibility/ 内复制需要修改的文件，并更新副本的脚本引用。

（1）复制 dom.html、dom-cases.js 为 practice-dom.html、practice-dom.js。修复第三组输入，让“开头”和“结尾”分别成段，&lt;div&gt; 保持独立。检查：两个非空段落与一个 &lt;div&gt; 同级，没有额外空段落；同步更新预期树。

（2）复制 whitespace.html 为 practice-whitespace.html，只移除两个 &lt;span&gt; 之间的换行。检查：childNodes 与 children 都变为 2；修改副本中相应计数检查及节点读取，解释为什么不能盲目删除整页空白。

（3）复制 xml.html 为 practice-xml.html，增加一组 XML 属性值未加引号的输入，再给出加引号的版本。检查：分别出现和不出现解析错误，原有斜线对照仍能执行。

（4）复制 quirks.html 为 practice-mode.html，只添加简短 DOCTYPE。检查：响应仍为 text/html，compatMode 由 BackCompat 变成 CSS1Compat；解释为何没有切换成 XML。

（5）选做：复制 migration.html 为 practice-migration.html，把提醒改成自己的主题，增加一条真正重要的截止信息。检查：语义与标题层级合理，Elements 中没有旧式元素，返回链接仍可用。

### 提示

第一题先确定三个同级元素的顺序；第二题不要继续假设第二个子节点是空白；第三题只改变引号，避免混入别的语法错误。练习副本用后可删除。

## 参考与引用来源

- WHATWG HTML：[解析模型与错误](https://html.spec.whatwg.org/multipage/parsing.html#parsing)、[in body 规则](https://html.spec.whatwg.org/multipage/parsing.html#parsing-main-inbody)，支持树构建与段落恢复；[HTML 语法](https://html.spec.whatwg.org/multipage/syntax.html#syntax)与[标签省略条件](https://html.spec.whatwg.org/multipage/syntax.html#optional-tags)，支持名称、属性与隐含元素；[布尔属性](https://html.spec.whatwg.org/multipage/common-microsyntaxes.html#boolean-attributes)、[XML 语法](https://html.spec.whatwg.org/multipage/xhtml.html#the-xhtml-syntax)、[过时特性](https://html.spec.whatwg.org/multipage/obsolete.html)；[文本级语义](https://html.spec.whatwg.org/multipage/text-level-semantics.html#text-level-semantics)中的 &lt;abbr&gt;、&lt;b&gt;、&lt;i&gt;、&lt;strong&gt;、&lt;em&gt;。
- W3C：[XML 1.0 第五版](https://www.w3.org/TR/xml/) §2.1、2.3、3.1，支持格式良好、名称匹配、属性引号与空元素标签。
- MDN：[DOMParser.parseFromString](https://developer.mozilla.org/en-US/docs/Web/API/DOMParser/parseFromString#description) 的类型参数、独立文档与错误处理；[空白处理](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Text/Whitespace#how_does_html_process_whitespace)、[childNodes](https://developer.mozilla.org/en-US/docs/Web/API/Node/childNodes)、[children](https://developer.mozilla.org/en-US/docs/Web/API/Element/children)，支持节点与排版的区别；[Media types](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/MIME_types)、[Document.contentType](https://developer.mozilla.org/en-US/docs/Web/API/Document/contentType)、[文档模式](https://developer.mozilla.org/en-US/docs/Web/HTML/Guides/Quirks_mode_and_standards_mode#how_do_browsers_determine_which_mode_to_use)与 [Document.compatMode](https://developer.mozilla.org/en-US/docs/Web/API/Document/compatMode#value)，支持 HTTP 类型、模式触发与读数限制；[outerHTML](https://developer.mozilla.org/en-US/docs/Web/API/Element/outerHTML#value)、[textContent](https://developer.mozilla.org/en-US/docs/Web/API/Node/textContent)、[JSON.stringify](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/JSON/stringify#description)，支持配套页的序列化与文本显示；[text-align](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/text-align#values)与 [color](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/color)，支持迁移示例样式。
- Chrome for Developers：[HTML versus the DOM](https://developer.chrome.com/docs/devtools/dom/#appendix)、[检查请求详情](https://developer.chrome.com/docs/devtools/network/#details)，支持源码、DOM 与响应头对照。
- Python 3.12：[http.server](https://docs.python.org/3.12/library/http.server.html#http.server.SimpleHTTPRequestHandler) 的 SimpleHTTPRequestHandler、send_head、ThreadingHTTPServer 与响应头方法，支持专用预览服务。